# 🚌 Bus Number Detection System

Detects bus numbers from CCTV video using **YOLOv8 + Tesseract OCR**  
and logs results to a live online dashboard.

### 👉 After running, view the live dashboard here:
## 🔴 [bus-monitor.up.railway.app](https://bus-monitor.up.railway.app)

---

### Steps (just click Runtime → Run All):
1. **Step 1** — Install dependencies + Tesseract OCR
2. **Step 2** — Configuration (pre-filled, no changes needed)
3. **Step 3** — Download video from Google Drive
4. **Step 4** — Download model from GitHub
5. **Step 5** — Verify Tesseract is working
6. **Step 6** — Run detection

> ⏱️ Takes ~5-10 minutes depending on video length

## Step 1 — Install Dependencies + Tesseract OCR

In [ ]:
# Install Python packages
!pip install -q ultralytics pytesseract mysql-connector-python gdown

# Install Tesseract OCR + English language data on the Colab machine
!apt-get install -q tesseract-ocr
!apt-get install -q tesseract-ocr-eng

# Confirm Tesseract installed correctly
!tesseract --version

# Create temp folder for cropped bus images
!mkdir -p ./temp_images

print('\n✅ All dependencies installed successfully')

## Step 2 — Configuration
> ✅ Pre-filled — no changes needed. Just run this cell.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
VIDEO_PATH     = 'input_video.mp4'
MODEL_PATH     = 'best.pt'
TESSERACT_PATH = '/usr/bin/tesseract'   # Tesseract location on Colab (do NOT change)
IMAGES_DIR     = './temp_images/'

# ── Google Drive File ID ───────────────────────────────────────────────
# Get from Drive share link: drive.google.com/file/d/THIS_PART/view
GOOGLE_DRIVE_FILE_ID = 'PASTE_YOUR_VIDEO_FILE_ID_HERE'

# ── GitHub repo details ────────────────────────────────────────────────
GITHUB_USERNAME = 'PASTE_YOUR_GITHUB_USERNAME_HERE'   # e.g. john123
GITHUB_REPO     = 'bus-monitoring-system'              # your repo name

# ── Detection settings ─────────────────────────────────────────────────
THRESHOLD         = 0.5
LINE_Y            = 800       # virtual line Y position — adjust for your video
VALID_BUS_NUMBERS = [9, 19, 15, 6, 5, 13, 7, 14]
LICENCE_PLATE_MAP = {
    19: 'TN 84 C35619',
     9: 'TN 84 A55709',
    15: 'TN 84 C75915',
     6: 'TN 84 C85806',
     5: 'TN 84 C35805',
    13: 'TN 84 C35913',
     7: 'TN 84 C25697',
    14: 'TN 84 C15514',
}

# ── Railway MySQL ──────────────────────────────────────────────────────
DB_HOST     = 'PASTE_RAILWAY_HOST_HERE'
DB_PORT     = 0000                        # paste your Railway port number
DB_USER     = 'root'
DB_PASSWORD = 'PASTE_RAILWAY_PASSWORD_HERE'
DB_NAME     = 'railway'

# ── Dashboard URL ──────────────────────────────────────────────────────
DASHBOARD_URL = 'https://bus-monitor.up.railway.app'  # your Railway dashboard URL

print('✅ Configuration loaded')

## Step 3 — Download Video from Google Drive

In [ ]:
import gdown, os

if GOOGLE_DRIVE_FILE_ID == 'PASTE_YOUR_VIDEO_FILE_ID_HERE':
    raise ValueError('❌ Please set your GOOGLE_DRIVE_FILE_ID in the Configuration cell!')

print('📥 Downloading video from Google Drive...')
url = f'https://drive.google.com/uc?id={GOOGLE_DRIVE_FILE_ID}'
gdown.download(url, VIDEO_PATH, quiet=False)

if os.path.exists(VIDEO_PATH):
    size = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f'✅ Video downloaded: {size:.1f} MB → {VIDEO_PATH}')
else:
    raise FileNotFoundError('❌ Video download failed. Check your File ID and make sure sharing is set to Anyone with link.')

## Step 4 — Download Model (best.pt) from GitHub

In [ ]:
import os

if GITHUB_USERNAME == 'PASTE_YOUR_GITHUB_USERNAME_HERE':
    raise ValueError('❌ Please set your GITHUB_USERNAME in the Configuration cell!')

if not os.path.exists(MODEL_PATH):
    print('📥 Downloading best.pt from GitHub...')
    github_url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/{GITHUB_REPO}/main/best.pt'
    result = os.system(f'wget -q --show-progress "{github_url}" -O {MODEL_PATH}')

    if result != 0 or not os.path.exists(MODEL_PATH):
        raise FileNotFoundError('❌ Model download failed. Make sure best.pt is committed to your GitHub repo.')

size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f'✅ Model ready: {size:.1f} MB → {MODEL_PATH}')

## Step 5 — Verify Tesseract is Working

In [ ]:
import pytesseract
from pytesseract import pytesseract as tess
import subprocess

# Set tesseract path
tess.tesseract_cmd = TESSERACT_PATH

# Verify tesseract binary exists
result = subprocess.run(['tesseract', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    version_line = result.stdout.split('\n')[0]
    print(f'✅ Tesseract found: {version_line}')
else:
    print('❌ Tesseract not found! Re-run Step 1.')

# Check English language data is available
langs = subprocess.run(['tesseract', '--list-langs'], capture_output=True, text=True)
if 'eng' in langs.stdout + langs.stderr:
    print('✅ English language data (eng) is available')
else:
    print('⚠️  English language data missing — installing now...')
    os.system('apt-get install -q tesseract-ocr-eng')
    print('✅ English language data installed')

print('\n✅ Tesseract is ready!')

## Step 6 — Run Detection
> 🚌 Results appear in the live dashboard as buses are detected!

In [ ]:
import cv2, datetime, os
import mysql.connector
from ultralytics import YOLO
from pytesseract import pytesseract as tess
from statistics import mode

# ── Setup ─────────────────────────────────────────────────────────────
tess.tesseract_cmd = TESSERACT_PATH
os.makedirs(IMAGES_DIR, exist_ok=True)

# Connect to Railway database
print('🔌 Connecting to Railway database...')
try:
    db = mysql.connector.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASSWORD,
        database=DB_NAME
    )
    cursor = db.cursor()
    print('✅ Database connected')
except Exception as e:
    raise ConnectionError(f'❌ Database connection failed: {e}\nCheck your Railway credentials in Step 2.')

# Load video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'❌ Cannot open video: {VIDEO_PATH}')
ret, frame = cap.read()
H, W, _ = frame.shape
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'📹 Video: {W}x{H} | {total_frames} frames')

# Load YOLO model
model = YOLO(MODEL_PATH)
print('✅ YOLO model loaded')
print(f'\n🚌 Starting detection...')
print(f'👉 Watch results live: {DASHBOARD_URL}')
print('-' * 60)

# ── State ──────────────────────────────────────────────────────────────
l             = []
saved_paths   = []
detected_in   = set()
detected_out  = set()
last_detected = 0
frame_num     = 0

# ── Main Loop ──────────────────────────────────────────────────────────
while ret:
    frame_num += 1

    # Show progress every 200 frames
    if frame_num % 200 == 0:
        pct = (frame_num / total_frames) * 100
        print(f'  ⏳ {frame_num}/{total_frames} frames ({pct:.0f}%)', end='\r')

    results = model(frame, verbose=False)[0]

    for result in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = result
        if score > THRESHOLD and y1 <= LINE_Y <= y2:
            if len(saved_paths) < 5:
                crop = frame[int(y1):int(y2), int(x1):int(x2)]
                path = os.path.join(IMAGES_DIR, f'crop_{len(saved_paths)}.jpg')
                cv2.imwrite(path, crop)
                saved_paths.append(path)

    if len(saved_paths) == 5:
        for p in saved_paths:
            img  = cv2.imread(p)
            text = tess.image_to_string(
                img,
                config='-l eng --psm 9 -c tessedit_char_whitelist=1234567890'
            )
            nums = [
                int(n) for n in text.split()
                if n.isdigit() and int(n) in VALID_BUS_NUMBERS
            ]
            l.extend(nums)

        saved_paths.clear()
        for f in os.scandir(IMAGES_DIR):
            if f.is_file(): os.remove(f.path)

        if l:
            bus_num = mode(l)
            now     = datetime.datetime.now()
            today   = datetime.date.today()
            plate   = LICENCE_PLATE_MAP.get(bus_num, 'Unknown')

            # Bus arriving IN
            if bus_num not in detected_in and bus_num not in detected_out:
                print(f'\n  ✅ BUS IN  | #{bus_num} | {plate} | {now.strftime("%H:%M:%S")}')
                cursor.execute(
                    "INSERT INTO bus_number_detection (bus_number, licence_plate_number, In_time, In_date) VALUES (%s,%s,%s,%s)",
                    (bus_num, plate, now, today)
                )
                db.commit()
                detected_in.add(bus_num)
                last_detected = bus_num

            # Bus going OUT
            elif bus_num in detected_in and last_detected != bus_num and bus_num not in detected_out:
                print(f'\n  🚌 BUS OUT | #{bus_num} | {now.strftime("%H:%M:%S")}')
                cursor.execute(
                    "UPDATE bus_number_detection SET Out_time=%s, Out_Date=%s WHERE bus_number=%s AND Out_time IS NULL",
                    (now, today, bus_num)
                )
                db.commit()
                detected_out.add(bus_num)
                detected_in.discard(bus_num)

        l.clear()

    ret, frame = cap.read()

# ── Done ───────────────────────────────────────────────────────────────
cap.release()
cursor.close()
db.close()

print(f'\n\n✅ Detection complete! Processed {frame_num} frames.')
print(f'\n👉 View all results on the live dashboard:')
print(f'   {DASHBOARD_URL}')

---
## 👀 View Live Dashboard

Click the link below to see all detected buses with in/out times:

## 🔴 [Open Live Dashboard](https://bus-monitor.up.railway.app)

The dashboard auto-refreshes every 10 seconds.